# Deliverable 2 · Asistente de normativa de Ingeniería UdeC
**Álvaro Contreras y Pablo Cortés · Qwen/Qwen3-4B · Google Colab T4**

Este notebook ejecuta baseline y RAG reales con prompts seleccionables y guarda las trazas. Las salidas anteriores se preservan en `Deliverable2_RAG/evidencia_historica/rag_qwen3_4b_original.ipynb`; las celdas actuales están sin ejecutar hasta una nueva corrida.

La auditoría de respuestas históricas es **88% RAG / 2% baseline** bajo el criterio de fuentes suficientes, pendiente de validación del equipo; E1 publicó 4%. No extrapolar esas métricas a cambios de corpus, prompt o modelo. Ver `PROTOCOLO_EVALUACION.md`.

En Colab: **Entorno de ejecución → Cambiar tipo → GPU T4**. Ejecutar preparación/carga antes de empezar a grabar. Las descargas requieren Internet; no forman parte de la latencia por pregunta.

## 1. Código de esta entrega
Si los cambios aún no están publicados en GitHub, elegir `FUENTE="zip"` y subir `genia_deliverable2.zip`. Contiene el código y los PDF, sin modelos. Tras publicar, se puede usar `FUENTE="github"` y un commit concreto en `REF_REPO`.

En una ejecución local dentro del repositorio se usa directamente la carpeta actual.

In [ ]:
from pathlib import Path
import os, subprocess, zipfile, sys

FUENTE = "zip"  # "zip" o "github" en Colab
REF_REPO = "main"  # usar el SHA del commit publicado para congelar una entrega
if Path("Deliverable2_RAG/pipeline_colab.py").exists():
    REPO = Path.cwd()
else:
    REPO = Path("/content/genia_entrega2")
    if FUENTE == "zip":
        from google.colab import files
        uploaded = files.upload()  # seleccionar genia_deliverable2.zip
        archive = next((Path(n) for n in uploaded if n.endswith(".zip")), None)
        if archive is None:
            raise ValueError("Debes subir el paquete ZIP de esta entrega")
        REPO.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.namelist():
                if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(REPO)
    elif FUENTE == "github":
        if not (REPO / ".git").exists():
            subprocess.run(["git", "clone", "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git", str(REPO)], check=True)
        subprocess.run(["git", "fetch", "origin", REF_REPO], cwd=REPO, check=True)
        subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
    else:
        raise ValueError("FUENTE debe ser zip o github")
    os.chdir(REPO)
if not Path("Deliverable2_RAG/pipeline_colab.py").exists():
    raise FileNotFoundError("El commit no contiene esta versión. Usa el ZIP o publica primero los cambios.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Código:", REPO)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "Deliverable2_RAG/requirements-colab.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Activa GPU T4 en Colab"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2. Auditoría histórica (sin inferencia)
Recuenta decisiones explícitas, vinculadas por hash a las respuestas guardadas. No ejecuta el modelo ni evalúa respuestas nuevas.

In [ ]:
from Deliverable2_RAG.auditoria import run, read_csv
import json
historico = run()
print(json.dumps(historico, ensure_ascii=False, indent=2))

## 3. Configuración y carga
`historico` conserva la base de 198 fragmentos utilizada en las respuestas previas. `v2` usa el extractor corregido (no confunde una referencia al Art. 7 con un encabezado dentro del Art. 9); sus resultados todavía no están medidos.

Se registran revisiones resueltas de ambos modelos, dtype efectivo, dispositivos, versiones, hashes y truncamiento del encoder. No se presupone que toda la carga esté en GPU ni una precisión específica.

`PROMPT_VERSION="completo_v1"` prueba instrucciones de completitud y citas; `historico` reproduce las instrucciones anteriores. Ninguna mejora del nuevo prompt ha sido medida todavía.


In [ ]:
from Deliverable2_RAG.pipeline_colab import load_pipeline, run_question, evaluate, new_run_dir, compare_prompts, select_prompt
CORPUS_VERSION = "historico"  # cambiar a "v2" solo para un experimento nuevo
PROMPT_VERSION = "completo_v1"  # o "historico"; el prompt nuevo requiere evaluación
K = 5
MODEL_REVISION = None  # copiar revision_modelo del metadata.json para repetir una corrida
EMBEDDING_REVISION = None
if CORPUS_VERSION not in {"historico", "v2"}:
    raise ValueError("Versión de corpus inválida")
corpus = Path("Deliverable2_RAG") / ("base_conocimiento_udec.json" if CORPUS_VERSION == "historico" else "base_conocimiento_v2.json")
if "pipeline" in globals():
    del pipeline
    import gc
    gc.collect()
    torch.cuda.empty_cache()
pipeline = load_pipeline(corpus, MODEL_REVISION, EMBEDDING_REVISION, PROMPT_VERSION)
questions = read_csv("test_set_50.csv")

## 3b. Comparación de prompts en los cinco casos revisados
Esta celda genera dos respuestas por pregunta con **idéntico contexto recuperado**. Solo cambia las instrucciones. Conserva el corpus histórico para esta primera comparación; después puede repetirse con v2. Son casos de desarrollo escogidos por sus fallos: no constituyen una estimación de mejora global.

Revisar si aparecen excepciones, todos los datos y sus fuentes. Si falta un artículo en el contexto, mejorar el prompt no garantiza resolver el caso. Ver `EXPERIMENTO_PROMPT.md`.


In [ ]:
ids_revision = {"24", "25", "36", "38", "40"}
casos_revision = [q for q in questions if q["id"] in ids_revision]
comparacion_dir = compare_prompts(pipeline, casos_revision, K)


## 4. Demostración en vivo: misma entrada, dos inferencias reales
Escribir una pregunta o dejar vacío para sortear una de las 50 al ejecutar. No hay respuestas pregrabadas en esta celda. El contexto mostrado es exactamente el que recibe RAG, con el mismo k. La traza queda guardada.

El tiempo de carga anterior no se incluye en la grabación de 3 minutos. No editar para ocultar la inferencia ni presentar la auditoría histórica como una nueva corrida.

In [ ]:
import secrets
PREGUNTA = ""  # también se puede escribir aquí una entrada propuesta por el evaluador
pregunta_demo = PREGUNTA.strip() or secrets.choice(questions)["pregunta"]
print("PREGUNTA:", pregunta_demo, flush=True)
demo_dir = new_run_dir()
(demo_dir / "metadata.json").write_text(json.dumps(pipeline["metadata"], ensure_ascii=False, indent=2))
for use_rag in (False, True):
    print("\nEJECUTANDO", "RAG" if use_rag else "BASELINE", flush=True)
    result = run_question(pipeline, pregunta_demo, use_rag, K)
    print(result["respuesta"])
    print(f"Latencia: {result['latencia_seg']:.2f} s")
    if result["contextos"]:
        print("CONTEXTO REAL ENVIADO AL MODELO:")
        for c in result["contextos"]:
            print(f"{c['similitud']:.4f} | {c['texto']}")
    (demo_dir / f"{result['sistema']}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
print("Evidencia:", demo_dir)

## 5. Evaluación completa: baseline y RAG sobre las 50 preguntas
Ejecutar antes del video; puede tardar varios minutos. Guarda predicciones crudas, contextos, latencias, hashes y una plantilla de revisión vacía. El gold solo se usa en el archivo de revisión: no se envía al modelo. Revisar con el mismo criterio ambos sistemas antes de calcular exactitud.

In [ ]:
select_prompt(pipeline, PROMPT_VERSION)  # cambia el prompt sin recargar el modelo
run_dir = evaluate(pipeline, questions, K)

## 6. Fallo real y diagnóstico
El caso P25 histórico omite qué asignaturas no pueden eliminarse. Primero se muestra la respuesta guardada y el corte del Art. 9. Luego puede ejecutarse nuevamente y revisar si el contexto contiene Art. 7 completo. Una salida correcta nueva no elimina el fallo histórico ni permite afirmar una mejora global.

In [ ]:
from Deliverable2_RAG.pipeline_colab import retrieve
p25 = questions[24]
old = read_csv("Deliverable2_RAG/resultados_rag_qwen3_4b.csv")[24]
print("CASO HISTÓRICO P25:", p25["pregunta"])
print("REFERENCIA:", p25["gold_dato"], "|", p25["gold_fuente"])
print("SALIDA HISTÓRICA:", old["prediccion_rag"])
old_chunks = json.loads(Path("Deliverable2_RAG/base_conocimiento_udec.json").read_text())
for c in old_chunks:
    if c["texto"].startswith("[Art. 9°, RI-FI]"):
        print("FRAGMENTO HISTÓRICO:", c["texto"])
print("\nNUEVA INFERENCIA (puede cambiar):", flush=True)
failure_probe = run_question(pipeline, p25["pregunta"], True, K)
print(failure_probe["respuesta"])
for c in failure_probe["contextos"]:
    print(c["texto"])
probe_dir = new_run_dir()
(probe_dir / "metadata.json").write_text(json.dumps(pipeline["metadata"], ensure_ascii=False, indent=2))
(probe_dir / "fallo_p25.json").write_text(json.dumps(failure_probe, ensure_ascii=False, indent=2))

## 7. Exportar evidencia
Descargar y conservar la carpeta de corrida, completar `revision_pendiente.csv` y guardar el notebook con sus salidas reales. No reemplazar los CSV históricos con resultados nuevos.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(run_dir), "zip", run_dir)
files.download(archive)
if "comparacion_dir" in globals():
    comparacion_zip = shutil.make_archive(str(comparacion_dir), "zip", comparacion_dir)
    files.download(comparacion_zip)
